# CodeMix-T — Phase 3: Training Loop

Covers:
1. Mount drive & reload architecture
2. Dataset class & DataLoader
3. Label-smoothed cross-entropy loss
4. Warmup + inverse-sqrt LR scheduler (Vaswani schedule)
5. Training step & validation step
6. Full training loop with checkpointing
7. WandB experiment tracking
8. Resume from checkpoint

## Cell 1 — Setup & Mount Drive

In [1]:
!pip install -q sentencepiece wandb

import torch, math, time, json, os
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from tqdm import tqdm
import sentencepiece as spm
import wandb

from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/CodeMixT'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Mounted at /content/drive
Device: cpu


## Cell 2 — Paste Architecture (copy all classes from Phase 2 here)

In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import numpy as np
from dataclasses import dataclass
from typing import Optional
import copy

In [9]:
# ── PASTE FULL ARCHITECTURE CODE FROM PHASE 2 HERE ──
# Copy everything from Cell 2 (CodeMixTConfig) through Cell 8 (masks)
# and the CodeMixT class from Cell 9.
#
# Then load config and model:

@dataclass
class CodeMixTConfig:
    """
    Central config for CodeMix-T architecture.
    All hyperparameters in one place — easy to swap between small/base.
    """
    # Vocabulary
    vocab_size: int     = 16000   # Must match your trained tokenizer
    pad_id: int         = 0
    bos_id: int         = 2
    eos_id: int         = 3

    # Language ID (novel component)
    num_languages: int  = 4       # EN=0, HI=1, TA=2, UNK=3
    lang_embed_dim: int = 64      # Dimension of language-ID embedding

    # Model dimensions
    d_model: int        = 512     # Main model dimension
    d_ff: int           = 2048    # Feed-forward hidden dimension
    num_heads: int      = 8       # Attention heads
    num_enc_layers: int = 6       # Encoder layers
    num_dec_layers: int = 6       # Decoder layers
    max_seq_len: int    = 128     # Max sequence length
    dropout: float      = 0.1

    # Inference
    beam_size: int      = 4
    max_gen_len: int    = 128

    def __post_init__(self):
        assert self.d_model % self.num_heads == 0, \
            f'd_model ({self.d_model}) must be divisible by num_heads ({self.num_heads})'
        self.d_k = self.d_model // self.num_heads  # Key/query dimension per head


# Two preset configs
CONFIG_SMALL = CodeMixTConfig(
    d_model=256, d_ff=1024, num_heads=4,
    num_enc_layers=4, num_dec_layers=4,
    lang_embed_dim=32
)

CONFIG_BASE = CodeMixTConfig()  # Default values above (~80M params)

# Use BASE for training, SMALL for quick debugging
cfg = CONFIG_BASE
print('Config loaded:')
print(f'  d_model={cfg.d_model}, heads={cfg.num_heads}, d_k={cfg.d_k}')
print(f'  enc_layers={cfg.num_enc_layers}, dec_layers={cfg.num_dec_layers}')
print(f'  vocab_size={cfg.vocab_size}, lang_embed_dim={cfg.lang_embed_dim}')


#cell3
class PositionalEncoding(nn.Module):
    """
    Sinusoidal positional encoding (Vaswani et al. 2017).
    Fixed (not learned) — encodes token position in sequence.

    PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))
    PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
    """
    def __init__(self, d_model: int, max_len: int, dropout: float):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        # Build PE matrix [max_len, d_model]
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)  # Even dims
        pe[:, 1::2] = torch.cos(position * div_term)  # Odd dims
        pe = pe.unsqueeze(0)  # [1, max_len, d_model]

        # Register as buffer (not a parameter, but saved with model)
        self.register_buffer('pe', pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [batch, seq_len, d_model]
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)


class CodeMixEmbedding(nn.Module):
    """
    Three-component embedding for CodeMix-T.

    Output = TokenEmbed(token_ids)        [batch, seq, d_model]
           + PosEncode(position)          [batch, seq, d_model]
           + LangIDEmbed(lang_ids) → proj [batch, seq, d_model]

    The Language-ID embedding is projected from lang_embed_dim → d_model
    via a learned linear layer before addition.
    """
    def __init__(self, cfg: CodeMixTConfig):
        super().__init__()

        # 1. Standard token embedding
        self.token_embed = nn.Embedding(
            cfg.vocab_size, cfg.d_model, padding_idx=cfg.pad_id
        )

        # 2. Sinusoidal positional encoding
        self.pos_encoding = PositionalEncoding(
            cfg.d_model, cfg.max_seq_len, cfg.dropout
        )

        # 3. Language-ID embedding (NOVEL COMPONENT)
        #    Maps language tag (0,1,2,3) → lang_embed_dim
        #    Then projects to d_model for addition
        self.lang_embed = nn.Embedding(cfg.num_languages, cfg.lang_embed_dim)
        self.lang_proj  = nn.Linear(cfg.lang_embed_dim, cfg.d_model, bias=False)

        self.dropout = nn.Dropout(cfg.dropout)

        # Scale token embeddings (standard practice from Vaswani et al.)
        self.scale = math.sqrt(cfg.d_model)

    def forward(
        self,
        token_ids: torch.Tensor,   # [batch, seq_len]
        lang_ids: torch.Tensor     # [batch, seq_len]  — Language ID per token
    ) -> torch.Tensor:
        # Token embedding scaled
        tok = self.token_embed(token_ids) * self.scale  # [B, S, d_model]

        # Language-ID embedding projected to d_model
        lang = self.lang_proj(self.lang_embed(lang_ids))  # [B, S, d_model]

        # Sum all three (positional encoding adds inside pos_encoding)
        x = tok + lang                    # Token + Language
        x = self.pos_encoding(x)          # + Positional
        return x                          # [B, S, d_model]


print('Embedding classes defined.')

# Quick shape test
_emb = CodeMixEmbedding(cfg)
_tok = torch.randint(0, cfg.vocab_size, (2, 10))
_lid = torch.randint(0, cfg.num_languages, (2, 10))
_out = _emb(_tok, _lid)
print(f'Embedding output shape: {_out.shape}  (expected [2, 10, {cfg.d_model}])')
del _emb, _tok, _lid, _out

#cell4
class MultiHeadAttention(nn.Module):
    """
    Multi-Head Scaled Dot-Product Attention.

    Attention(Q,K,V) = softmax(QK^T / sqrt(d_k)) * V

    Used for:
      - Encoder self-attention
      - Decoder masked self-attention  (causal mask)
      - Decoder cross-attention        (attends to encoder output)
    """
    def __init__(self, cfg: CodeMixTConfig):
        super().__init__()
        self.num_heads = cfg.num_heads
        self.d_k       = cfg.d_k
        self.d_model   = cfg.d_model

        # Projection matrices for Q, K, V, and output
        self.W_q = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.W_k = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.W_v = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.W_o = nn.Linear(cfg.d_model, cfg.d_model, bias=False)

        self.dropout = nn.Dropout(cfg.dropout)
        self.scale   = math.sqrt(self.d_k)

    def split_heads(self, x: torch.Tensor) -> torch.Tensor:
        """[B, S, d_model] → [B, heads, S, d_k]"""
        B, S, _ = x.shape
        x = x.view(B, S, self.num_heads, self.d_k)
        return x.transpose(1, 2)  # [B, heads, S, d_k]

    def merge_heads(self, x: torch.Tensor) -> torch.Tensor:
        """[B, heads, S, d_k] → [B, S, d_model]"""
        B, _, S, _ = x.shape
        x = x.transpose(1, 2).contiguous()
        return x.view(B, S, self.d_model)

    def forward(
        self,
        query: torch.Tensor,                # [B, S_q, d_model]
        key:   torch.Tensor,                # [B, S_k, d_model]
        value: torch.Tensor,                # [B, S_k, d_model]
        mask:  Optional[torch.Tensor]=None  # Attention mask
    ) -> torch.Tensor:

        Q = self.split_heads(self.W_q(query))  # [B, h, S_q, d_k]
        K = self.split_heads(self.W_k(key))    # [B, h, S_k, d_k]
        V = self.split_heads(self.W_v(value))  # [B, h, S_k, d_k]

        # Scaled dot-product attention
        scores = torch.matmul(Q, K.transpose(-2, -1)) / self.scale  # [B, h, S_q, S_k]

        if mask is not None:
            # Fill masked positions with -inf so softmax → ~0
            scores = scores.masked_fill(mask == 0, float('-inf'))

        attn_weights = self.dropout(F.softmax(scores, dim=-1))  # [B, h, S_q, S_k]

        # Weighted sum of values
        context = torch.matmul(attn_weights, V)   # [B, h, S_q, d_k]
        context = self.merge_heads(context)        # [B, S_q, d_model]

        return self.W_o(context)                   # [B, S_q, d_model]


print('MultiHeadAttention defined.')

# Shape test
_mha = MultiHeadAttention(cfg)
_q = torch.randn(2, 10, cfg.d_model)
_k = torch.randn(2, 15, cfg.d_model)
_v = torch.randn(2, 15, cfg.d_model)
_out = _mha(_q, _k, _v)
print(f'MHA output shape: {_out.shape}  (expected [2, 10, {cfg.d_model}])')
del _mha, _q, _k, _v, _out
#cell5
class PositionwiseFeedForward(nn.Module):
    """
    FFN(x) = max(0, xW1 + b1)W2 + b2
    Applied independently to each position.
    d_model → d_ff → d_model
    """
    def __init__(self, cfg: CodeMixTConfig):
        super().__init__()
        self.linear1 = nn.Linear(cfg.d_model, cfg.d_ff)
        self.linear2 = nn.Linear(cfg.d_ff, cfg.d_model)
        self.dropout = nn.Dropout(cfg.dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, S, d_model]
        return self.linear2(self.dropout(F.gelu(self.linear1(x))))
        # Note: using GELU instead of ReLU — smoother gradients, used in BERT/GPT


print('PositionwiseFeedForward defined.')

#cell6
class EncoderLayer(nn.Module):
    """
    Single Transformer encoder layer.

    Using Pre-LN (Layer Norm before sublayer) for more stable training:
        x = x + SelfAttn(LayerNorm(x))
        x = x + FFN(LayerNorm(x))

    (Post-LN = original Vaswani paper; Pre-LN = used in GPT-2, more stable)
    """
    def __init__(self, cfg: CodeMixTConfig):
        super().__init__()
        self.self_attn = MultiHeadAttention(cfg)
        self.ffn       = PositionwiseFeedForward(cfg)
        self.norm1     = nn.LayerNorm(cfg.d_model)
        self.norm2     = nn.LayerNorm(cfg.d_model)
        self.dropout   = nn.Dropout(cfg.dropout)

    def forward(
        self,
        x:    torch.Tensor,                  # [B, S, d_model]
        mask: Optional[torch.Tensor] = None  # Padding mask
    ) -> torch.Tensor:
        # Pre-LN self-attention + residual
        residual = x
        x = self.norm1(x)
        x = residual + self.dropout(self.self_attn(x, x, x, mask))

        # Pre-LN FFN + residual
        residual = x
        x = self.norm2(x)
        x = residual + self.dropout(self.ffn(x))

        return x  # [B, S, d_model]


class Encoder(nn.Module):
    """
    Full encoder: Embedding → N × EncoderLayer → LayerNorm
    """
    def __init__(self, cfg: CodeMixTConfig):
        super().__init__()
        self.embedding = CodeMixEmbedding(cfg)
        self.layers    = nn.ModuleList([EncoderLayer(cfg) for _ in range(cfg.num_enc_layers)])
        self.norm      = nn.LayerNorm(cfg.d_model)

    def forward(
        self,
        token_ids: torch.Tensor,  # [B, src_len]
        lang_ids:  torch.Tensor,  # [B, src_len]
        src_mask:  Optional[torch.Tensor] = None
    ) -> torch.Tensor:
        x = self.embedding(token_ids, lang_ids)  # [B, src_len, d_model]
        for layer in self.layers:
            x = layer(x, src_mask)
        return self.norm(x)  # [B, src_len, d_model]


print('EncoderLayer and Encoder defined.')

# Shape test
_enc = Encoder(cfg)
_tok = torch.randint(0, cfg.vocab_size, (2, 12))
_lid = torch.randint(0, cfg.num_languages, (2, 12))
_out = _enc(_tok, _lid)
print(f'Encoder output shape: {_out.shape}  (expected [2, 12, {cfg.d_model}])')
del _enc, _tok, _lid, _out

#cell7
class DecoderLayer(nn.Module):
    """
    Single Transformer decoder layer — three sublayers:
      1. Masked self-attention  (causal: can't see future tokens)
      2. Cross-attention        (attends to encoder memory)
      3. Feed-forward network

    Pre-LN variant for training stability.
    """
    def __init__(self, cfg: CodeMixTConfig):
        super().__init__()
        self.self_attn  = MultiHeadAttention(cfg)  # Masked self-attn
        self.cross_attn = MultiHeadAttention(cfg)  # Cross-attn over encoder
        self.ffn        = PositionwiseFeedForward(cfg)
        self.norm1      = nn.LayerNorm(cfg.d_model)
        self.norm2      = nn.LayerNorm(cfg.d_model)
        self.norm3      = nn.LayerNorm(cfg.d_model)
        self.dropout    = nn.Dropout(cfg.dropout)

    def forward(
        self,
        x:          torch.Tensor,                   # [B, tgt_len, d_model]
        enc_output: torch.Tensor,                   # [B, src_len, d_model]
        tgt_mask:   Optional[torch.Tensor] = None,  # Causal mask
        src_mask:   Optional[torch.Tensor] = None   # Padding mask
    ) -> torch.Tensor:

        # 1. Masked self-attention
        residual = x
        x = self.norm1(x)
        x = residual + self.dropout(self.self_attn(x, x, x, tgt_mask))

        # 2. Cross-attention (Q from decoder, K/V from encoder)
        residual = x
        x = self.norm2(x)
        x = residual + self.dropout(self.cross_attn(x, enc_output, enc_output, src_mask))

        # 3. Feed-forward
        residual = x
        x = self.norm3(x)
        x = residual + self.dropout(self.ffn(x))

        return x  # [B, tgt_len, d_model]


class Decoder(nn.Module):
    """
    Full decoder: Embedding → N × DecoderLayer → LayerNorm
    Note: Decoder uses standard token embedding only (no lang-ID needed
    for the English output side — English is always LANG_EN).
    """
    def __init__(self, cfg: CodeMixTConfig):
        super().__init__()
        # Standard embedding for target (English) — no language-ID needed
        self.token_embed  = nn.Embedding(cfg.vocab_size, cfg.d_model, padding_idx=cfg.pad_id)
        self.pos_encoding = PositionalEncoding(cfg.d_model, cfg.max_seq_len, cfg.dropout)
        self.layers       = nn.ModuleList([DecoderLayer(cfg) for _ in range(cfg.num_dec_layers)])
        self.norm         = nn.LayerNorm(cfg.d_model)
        self.scale        = math.sqrt(cfg.d_model)

    def forward(
        self,
        tgt_ids:    torch.Tensor,                   # [B, tgt_len]
        enc_output: torch.Tensor,                   # [B, src_len, d_model]
        tgt_mask:   Optional[torch.Tensor] = None,
        src_mask:   Optional[torch.Tensor] = None
    ) -> torch.Tensor:
        x = self.token_embed(tgt_ids) * self.scale  # [B, tgt_len, d_model]
        x = self.pos_encoding(x)
        for layer in self.layers:
            x = layer(x, enc_output, tgt_mask, src_mask)
        return self.norm(x)  # [B, tgt_len, d_model]


print('DecoderLayer and Decoder defined.')

#cell8
def make_src_mask(src: torch.Tensor, pad_id: int) -> torch.Tensor:
    """
    Source padding mask — hides PAD tokens from attention.
    Returns: [B, 1, 1, src_len]  (broadcast over heads and query positions)
    """
    return (src != pad_id).unsqueeze(1).unsqueeze(2)  # [B, 1, 1, src_len]


def make_tgt_mask(tgt: torch.Tensor, pad_id: int) -> torch.Tensor:
    """
    Target causal mask — combines:
      1. Causal (lower-triangular) mask: token i can't attend to j > i
      2. Padding mask: ignore PAD tokens

    Returns: [B, 1, tgt_len, tgt_len]
    """
    B, T = tgt.shape

    # Padding mask: [B, 1, 1, T]
    pad_mask = (tgt != pad_id).unsqueeze(1).unsqueeze(2)

    # Causal mask: [1, 1, T, T]  — lower triangular
    causal_mask = torch.tril(torch.ones(T, T, device=tgt.device)).bool()
    causal_mask = causal_mask.unsqueeze(0).unsqueeze(0)

    # Combine: both conditions must be True
    return pad_mask & causal_mask  # [B, 1, T, T]


# Test masks
_src = torch.tensor([[5, 10, 3, 0, 0]])      # Last 2 are PAD
_tgt = torch.tensor([[2, 5, 10, 3]])          # <bos> token token <eos>

src_m = make_src_mask(_src, pad_id=0)
tgt_m = make_tgt_mask(_tgt, pad_id=0)

print(f'src_mask shape: {src_m.shape}')  # [1, 1, 1, 5]
print(f'src_mask:       {src_m}')
print(f'tgt_mask shape: {tgt_m.shape}')  # [1, 1, 4, 4]
print(f'tgt_mask (causal):\n{tgt_m[0,0]}')
del _src, _tgt, src_m, tgt_m

#cell9
class CodeMixT(nn.Module):
    """
    CodeMix-T: Full encoder-decoder Transformer for code-mixed translation.

    Novel features:
      - Language-ID-Aware Embeddings in the encoder
      - Trained from scratch on Tanglish + Hinglish corpora
      - Custom BPE tokenizer covering mixed-script tokens
    """
    def __init__(self, cfg: CodeMixTConfig):
        super().__init__()
        self.cfg     = cfg
        self.encoder = Encoder(cfg)
        self.decoder = Decoder(cfg)

        # Final projection: d_model → vocab_size
        self.output_projection = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)

        # Weight tying: share decoder embedding weights with output projection
        # (standard practice — reduces parameters, improves performance)
        self.output_projection.weight = self.decoder.token_embed.weight

        # Initialize weights
        self._init_weights()

    def _init_weights(self):
        """Xavier uniform init for linear layers, normal for embeddings."""
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Embedding):
                nn.init.normal_(module.weight, mean=0, std=0.02)
                if module.padding_idx is not None:
                    module.weight.data[module.padding_idx].zero_()

    def encode(
        self,
        src_ids:  torch.Tensor,  # [B, src_len]
        lang_ids: torch.Tensor,  # [B, src_len]
        src_mask: Optional[torch.Tensor] = None
    ) -> torch.Tensor:
        """Encode source sequence → encoder memory."""
        return self.encoder(src_ids, lang_ids, src_mask)

    def decode(
        self,
        tgt_ids:    torch.Tensor,
        enc_output: torch.Tensor,
        tgt_mask:   Optional[torch.Tensor] = None,
        src_mask:   Optional[torch.Tensor] = None
    ) -> torch.Tensor:
        """Decode target sequence given encoder memory."""
        dec_out = self.decoder(tgt_ids, enc_output, tgt_mask, src_mask)
        return self.output_projection(dec_out)  # [B, tgt_len, vocab_size]

    def forward(
        self,
        src_ids:  torch.Tensor,  # [B, src_len]
        lang_ids: torch.Tensor,  # [B, src_len]
        tgt_ids:  torch.Tensor,  # [B, tgt_len]
    ) -> torch.Tensor:
        """
        Full forward pass for training.
        Returns logits: [B, tgt_len, vocab_size]
        """
        src_mask = make_src_mask(src_ids, self.cfg.pad_id).to(src_ids.device)
        tgt_mask = make_tgt_mask(tgt_ids, self.cfg.pad_id).to(tgt_ids.device)

        enc_output = self.encode(src_ids, lang_ids, src_mask)
        logits     = self.decode(tgt_ids, enc_output, tgt_mask, src_mask)
        return logits  # [B, tgt_len, vocab_size]

    def count_parameters(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


# Instantiate and check
model = CodeMixT(cfg).to(device)
n_params = model.count_parameters()
print(f'CodeMix-T instantiated successfully.')
print(f'Total trainable parameters: {n_params:,}  ({n_params/1e6:.1f}M)')

# Forward pass test
B, SRC_LEN, TGT_LEN = 2, 20, 15
_src  = torch.randint(1, cfg.vocab_size, (B, SRC_LEN)).to(device)
_lang = torch.randint(0, cfg.num_languages, (B, SRC_LEN)).to(device)
_tgt  = torch.randint(1, cfg.vocab_size, (B, TGT_LEN)).to(device)

with torch.no_grad():
    logits = model(_src, _lang, _tgt)

print(f'Forward pass output shape: {logits.shape}  (expected [{B}, {TGT_LEN}, {cfg.vocab_size}])')
del _src, _lang, _tgt

#suiiii
with open(f'{DRIVE_DIR}/model/config.json') as f:
    cfg_dict = json.load(f)

from dataclasses import dataclass
# (assuming CodeMixTConfig is pasted above)
cfg = CodeMixTConfig(**cfg_dict)

model = CodeMixT(cfg).to(device)
model.load_state_dict(torch.load(f'{DRIVE_DIR}/model/codemix_t_init.pt', map_location=device))
print(f'Model loaded: {model.count_parameters()/1e6:.1f}M params')

Config loaded:
  d_model=512, heads=8, d_k=64
  enc_layers=6, dec_layers=6
  vocab_size=16000, lang_embed_dim=64
Embedding classes defined.
Embedding output shape: torch.Size([2, 10, 512])  (expected [2, 10, 512])
MultiHeadAttention defined.
MHA output shape: torch.Size([2, 10, 512])  (expected [2, 10, 512])
PositionwiseFeedForward defined.
EncoderLayer and Encoder defined.
Encoder output shape: torch.Size([2, 12, 512])  (expected [2, 12, 512])
DecoderLayer and Decoder defined.
src_mask shape: torch.Size([1, 1, 1, 5])
src_mask:       tensor([[[[ True,  True,  True, False, False]]]])
tgt_mask shape: torch.Size([1, 1, 4, 4])
tgt_mask (causal):
tensor([[ True, False, False, False],
        [ True,  True, False, False],
        [ True,  True,  True, False],
        [ True,  True,  True,  True]])
CodeMix-T instantiated successfully.
Total trainable parameters: 60,520,704  (60.5M)
Forward pass output shape: torch.Size([2, 15, 16000])  (expected [2, 15, 16000])
Model loaded: 60.5M params


## Cell 3 — Dataset Class

In [10]:
class CodeMixDataset(Dataset):
    """
    Loads pre-tokenized JSONL files from Phase 1.
    Each record has: src_ids, tgt_ids, source_lang_ids
    """
    def __init__(self, jsonl_path: str, max_src_len: int = 128, max_tgt_len: int = 128):
        self.samples = []
        self.max_src = max_src_len
        self.max_tgt = max_tgt_len

        with open(jsonl_path) as f:
            for line in f:
                rec = json.loads(line)
                src = rec['src_ids']
                tgt = rec['tgt_ids']
                lid = list(map(int, rec['source_lang_ids'].split()))

                # Align lang_ids with tokenized src (best effort)
                if len(lid) < len(src):
                    lid = lid + [0] * (len(src) - len(lid))
                lid = lid[:len(src)]

                if 3 <= len(src) <= max_src_len and 3 <= len(tgt) <= max_tgt_len:
                    self.samples.append((src, tgt, lid))

        print(f'Loaded {len(self.samples):,} samples from {Path(jsonl_path).name}')

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        src, tgt, lid = self.samples[idx]
        return (
            torch.tensor(src, dtype=torch.long),
            torch.tensor(tgt, dtype=torch.long),
            torch.tensor(lid, dtype=torch.long)
        )


def collate_fn(batch, pad_id=0):
    """Pad sequences in a batch to the same length."""
    srcs, tgts, lids = zip(*batch)

    src_lens = [s.size(0) for s in srcs]
    tgt_lens = [t.size(0) for t in tgts]
    max_src  = max(src_lens)
    max_tgt  = max(tgt_lens)

    src_pad  = torch.full((len(srcs), max_src), pad_id, dtype=torch.long)
    tgt_pad  = torch.full((len(tgts), max_tgt), pad_id, dtype=torch.long)
    lid_pad  = torch.zeros(len(lids), max_src, dtype=torch.long)

    for i, (s, t, l) in enumerate(zip(srcs, tgts, lids)):
        src_pad[i, :s.size(0)] = s
        tgt_pad[i, :t.size(0)] = t
        lid_pad[i, :l.size(0)] = l

    return src_pad, tgt_pad, lid_pad


# Load datasets
train_ds = CodeMixDataset(f'{DRIVE_DIR}/data_final/train.jsonl')
val_ds   = CodeMixDataset(f'{DRIVE_DIR}/data_final/val.jsonl')

BATCH_SIZE = 32
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=lambda b: collate_fn(b, cfg.pad_id), num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=lambda b: collate_fn(b, cfg.pad_id), num_workers=2)

print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')

Loaded 26,084 samples from train.jsonl
Loaded 1,449 samples from val.jsonl
Train batches: 816 | Val batches: 46


## Cell 4 — Label-Smoothed Cross Entropy Loss

In [11]:
class LabelSmoothingLoss(nn.Module):
    """
    Cross-entropy with label smoothing.
    Instead of hard 0/1 targets, smooth to (eps/V) for wrong classes
    and (1 - eps + eps/V) for the correct class.
    Prevents overconfidence and improves generalisation (Szegedy et al. 2016).
    eps=0.1 is standard for translation tasks.
    """
    def __init__(self, vocab_size: int, pad_id: int, smoothing: float = 0.1):
        super().__init__()
        self.vocab_size = vocab_size
        self.pad_id     = pad_id
        self.smoothing  = smoothing
        self.confidence = 1.0 - smoothing

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        # logits:  [B * T, vocab_size]
        # targets: [B * T]
        B = logits.size(0)
        log_probs = F.log_softmax(logits, dim=-1)

        # Smoothed target distribution
        smooth_val = self.smoothing / (self.vocab_size - 2)  # -2 for pad and true class
        with torch.no_grad():
            dist = torch.full_like(log_probs, smooth_val)
            dist.scatter_(1, targets.unsqueeze(1), self.confidence)
            dist[:, self.pad_id] = 0  # Never predict PAD
            mask = (targets == self.pad_id)
            dist[mask] = 0            # Zero out pad positions

        loss = -(dist * log_probs).sum(dim=-1)
        non_pad = (~mask).sum()
        return loss.sum() / non_pad.clamp(min=1)


criterion = LabelSmoothingLoss(cfg.vocab_size, cfg.pad_id, smoothing=0.1)
print('LabelSmoothingLoss defined.')

LabelSmoothingLoss defined.


## Cell 5 — Optimizer & LR Scheduler (Vaswani et al. warmup schedule)

In [12]:
class WarmupInvSqrtScheduler:
    """
    Learning rate schedule from 'Attention Is All You Need'.
    lr = d_model^-0.5 * min(step^-0.5, step * warmup_steps^-1.5)

    - Linearly increases LR during warmup
    - Decreases proportionally to inverse sqrt of step after warmup
    - warmup_steps=4000 is the standard value
    """
    def __init__(self, optimizer, d_model: int, warmup_steps: int = 4000):
        self.optimizer    = optimizer
        self.d_model      = d_model
        self.warmup_steps = warmup_steps
        self.step_num     = 0

    def step(self):
        self.step_num += 1
        lr = self._get_lr()
        for pg in self.optimizer.param_groups:
            pg['lr'] = lr
        return lr

    def _get_lr(self):
        s = self.step_num
        w = self.warmup_steps
        return (self.d_model ** -0.5) * min(s ** -0.5, s * w ** -1.5)

    def state_dict(self):
        return {'step_num': self.step_num}

    def load_state_dict(self, sd):
        self.step_num = sd['step_num']


optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1.0,             # Scheduler controls the actual LR
    betas=(0.9, 0.98),
    eps=1e-9
)
scheduler = WarmupInvSqrtScheduler(optimizer, d_model=cfg.d_model, warmup_steps=4000)

print('Optimizer and scheduler ready.')

# Visualise the LR schedule
steps = list(range(1, 20001))
_sched = WarmupInvSqrtScheduler(torch.optim.Adam([torch.zeros(1)]), cfg.d_model, 4000)
lrs = []
for s in steps:
    _sched.step_num = s
    lrs.append(_sched._get_lr())
peak_step = lrs.index(max(lrs)) + 1
print(f'Peak LR = {max(lrs):.6f} at step {peak_step}')
print(f'LR at step 1000  = {lrs[999]:.6f}')
print(f'LR at step 10000 = {lrs[9999]:.6f}')

Optimizer and scheduler ready.
Peak LR = 0.000699 at step 4000
LR at step 1000  = 0.000175
LR at step 10000 = 0.000442


## Cell 6 — Training & Validation Steps

In [13]:
def train_step(model, batch, optimizer, scheduler, criterion, device, grad_clip=1.0):
    """
    Single training step.
    Uses teacher forcing: feed ground-truth target tokens as decoder input.
    """
    model.train()
    src_ids, tgt_ids, lang_ids = [x.to(device) for x in batch]

    # Teacher forcing:
    # Decoder input  = tgt[:-1]  (all tokens except last)
    # Decoder target = tgt[1:]   (all tokens except first BOS)
    dec_input  = tgt_ids[:, :-1]
    dec_target = tgt_ids[:, 1:]

    # Forward pass
    logits = model(src_ids, lang_ids, dec_input)  # [B, T-1, vocab]

    # Compute loss
    B, T, V = logits.shape
    loss = criterion(logits.reshape(B*T, V), dec_target.reshape(B*T))

    # Backprop
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
    optimizer.step()
    lr = scheduler.step()

    return loss.item(), lr


@torch.no_grad()
def val_step(model, val_loader, criterion, device):
    """Full validation pass — returns avg loss and perplexity."""
    model.eval()
    total_loss, n_batches = 0.0, 0

    for batch in val_loader:
        src_ids, tgt_ids, lang_ids = [x.to(device) for x in batch]
        dec_input  = tgt_ids[:, :-1]
        dec_target = tgt_ids[:, 1:]

        logits = model(src_ids, lang_ids, dec_input)
        B, T, V = logits.shape
        loss = criterion(logits.reshape(B*T, V), dec_target.reshape(B*T))
        total_loss += loss.item()
        n_batches  += 1

    avg_loss   = total_loss / n_batches
    perplexity = math.exp(min(avg_loss, 20))  # Clip to avoid overflow
    return avg_loss, perplexity


print('train_step and val_step defined.')

train_step and val_step defined.


## Cell 7 — Checkpoint Utilities

In [14]:
CKPT_DIR = f'{DRIVE_DIR}/checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)


def save_checkpoint(model, optimizer, scheduler, epoch, step, val_loss, tag='latest'):
    path = f'{CKPT_DIR}/codemix_t_{tag}.pt'
    torch.save({
        'epoch':      epoch,
        'step':       step,
        'val_loss':   val_loss,
        'model':      model.state_dict(),
        'optimizer':  optimizer.state_dict(),
        'scheduler':  scheduler.state_dict(),
    }, path)
    print(f'Checkpoint saved: {path}  (val_loss={val_loss:.4f})')


def load_checkpoint(model, optimizer, scheduler, tag='latest'):
    path = f'{CKPT_DIR}/codemix_t_{tag}.pt'
    if not os.path.exists(path):
        print(f'No checkpoint found at {path}')
        return 0, 0, float('inf')

    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt['model'])
    optimizer.load_state_dict(ckpt['optimizer'])
    scheduler.load_state_dict(ckpt['scheduler'])
    print(f'Loaded checkpoint: epoch={ckpt["epoch"]}, step={ckpt["step"]}, val_loss={ckpt["val_loss"]:.4f}')
    return ckpt['epoch'], ckpt['step'], ckpt['val_loss']


print('Checkpoint utilities ready.')

Checkpoint utilities ready.


## Cell 8 — WandB Initialisation

In [15]:
# Create a free WandB account at wandb.ai, then paste your API key when prompted
wandb.login()

run = wandb.init(
    project='codemix-t',
    name='base-run-1',
    config={
        'd_model':        cfg.d_model,
        'd_ff':           cfg.d_ff,
        'num_heads':      cfg.num_heads,
        'enc_layers':     cfg.num_enc_layers,
        'dec_layers':     cfg.num_dec_layers,
        'vocab_size':     cfg.vocab_size,
        'lang_embed_dim': cfg.lang_embed_dim,
        'batch_size':     BATCH_SIZE,
        'warmup_steps':   4000,
        'label_smoothing': 0.1,
        'grad_clip':      1.0,
        'total_params':   model.count_parameters(),
    }
)
print(f'WandB run: {run.name}')

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 1


wandb: You chose 'Create a W&B account'
wandb: Create an account here: https://wandb.ai/authorize?signup=true&ref=models
wandb: After creating your account, create a new API key and store it securely.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: srushtirm-btech23 (srushtirm-btech23-rv-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


WandB run: base-run-1


## Cell 9 — Full Training Loop

In [ ]:
NUM_EPOCHS      = 30
VAL_EVERY       = 1          # Validate every N epochs
SAVE_EVERY      = 1          # Save checkpoint every N epochs
LOG_EVERY       = 50         # Log to WandB every N steps
PATIENCE        = 5          # Early stopping patience

# Optionally resume from checkpoint
start_epoch, global_step, best_val_loss = load_checkpoint(model, optimizer, scheduler)

no_improve = 0

for epoch in range(start_epoch, NUM_EPOCHS):
    model.train()
    epoch_loss = 0.0
    t0 = time.time()

    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{NUM_EPOCHS}')
    for batch in pbar:
        loss, lr = train_step(model, batch, optimizer, scheduler, criterion, device)
        epoch_loss  += loss
        global_step += 1

        pbar.set_postfix({'loss': f'{loss:.4f}', 'lr': f'{lr:.2e}'})

        if global_step % LOG_EVERY == 0:
            wandb.log({'train/loss': loss, 'train/lr': lr, 'step': global_step})

    avg_train_loss = epoch_loss / len(train_loader)
    epoch_time     = time.time() - t0

    # Validation
    if (epoch + 1) % VAL_EVERY == 0:
        val_loss, val_ppl = val_step(model, val_loader, criterion, device)

        print(f'Epoch {epoch+1:3d} | '
              f'train_loss={avg_train_loss:.4f} | '
              f'val_loss={val_loss:.4f} | '
              f'val_ppl={val_ppl:.2f} | '
              f'time={epoch_time:.0f}s')

        wandb.log({
            'epoch':      epoch + 1,
            'train/epoch_loss': avg_train_loss,
            'val/loss':   val_loss,
            'val/ppl':    val_ppl,
        })

        # Save best model
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            save_checkpoint(model, optimizer, scheduler, epoch+1, global_step, val_loss, tag='best')
            no_improve = 0
        else:
            no_improve += 1
            print(f'  No improvement for {no_improve}/{PATIENCE} epochs')

        # Early stopping
        if no_improve >= PATIENCE:
            print(f'Early stopping at epoch {epoch+1}')
            break

    # Periodic checkpoint
    if (epoch + 1) % SAVE_EVERY == 0:
        save_checkpoint(model, optimizer, scheduler, epoch+1, global_step, val_loss, tag='latest')

print(f'\nTraining complete. Best val_loss = {best_val_loss:.4f}')
wandb.finish()

No checkpoint found at /content/drive/MyDrive/CodeMixT/checkpoints/codemix_t_latest.pt


Epoch 1/30:   1%|          | 8/816 [03:19<5:32:28, 24.69s/it, loss=304.1170, lr=1.40e-06]